In [ ]:
# Historical standalone prototype. The maintained app uses pipeline/.
from pathlib import Path
import sys
project_root = Path.cwd() if (Path.cwd() / "config.py").is_file() else Path.cwd().parent
sys.path.insert(0, str(project_root))
import config

import cv2
import numpy as np
from ultralytics import YOLO
from collections import deque
import os

# ----------------------------
# Load YOLOv8 model
# ----------------------------
model = YOLO(str(config.MODEL_PATH))

# ----------------------------
# Create output folder
# ----------------------------
output_dir = str(config.OUTPUT_FOLDER / "prototypes")
os.makedirs(output_dir, exist_ok=True)

# ----------------------------
# Video input
# ----------------------------
video_path = str(config.DEFAULT_VIDEO_PATH)
cap = cv2.VideoCapture(video_path)

width  = int(cap.get(cv2.CAP_PROP_FRAME_WIDTH))
height = int(cap.get(cv2.CAP_PROP_FRAME_HEIGHT))
fps    = int(cap.get(cv2.CAP_PROP_FPS))
if fps == 0:
    fps = 30

# ----------------------------
# Video writer
# ----------------------------
output_path = os.path.join(output_dir, "output.avi")
fourcc = cv2.VideoWriter_fourcc(*'XVID')
out = cv2.VideoWriter(output_path, fourcc, fps, (width, height))

print("Saving output to:", output_path)

# ----------------------------
# Kalman Filter (x, y, vx, vy)
# ----------------------------
kalman = cv2.KalmanFilter(4, 2)

kalman.measurementMatrix = np.array([
    [1, 0, 0, 0],
    [0, 1, 0, 0]
], np.float32)

kalman.transitionMatrix = np.array([
    [1, 0, 1, 0],
    [0, 1, 0, 1],
    [0, 0, 1, 0],
    [0, 0, 0, 1]
], np.float32)

kalman.processNoiseCov = np.eye(4, dtype=np.float32) * 0.03
kalman.measurementNoiseCov = np.eye(2, dtype=np.float32) * 0.5

# ----------------------------
# Tracking variables
# ----------------------------
points = deque(maxlen=30)
initialized = False
prediction = np.zeros((4,1), dtype=np.float32)

# ----------------------------
# Main loop
# ----------------------------
while True:
    ret, frame = cap.read()
    if not ret:
        break

    # ----------------------------
    # YOLO detection (FULL FRAME)
    # ----------------------------
    results = model(frame, classes=[k for k, v in model.names.items() if v == "golf_ball"])[0]

    best_center = None
    best_conf = 0

    for box in results.boxes:
        cls = int(box.cls[0])
        conf = float(box.conf[0])

        if conf < 0.2:
            continue

        x1, y1, x2, y2 = map(int, box.xyxy[0])
        label = model.names[cls]

        # Draw detection box
        cv2.rectangle(frame, (x1, y1), (x2, y2), (0,255,0), 2)
        cv2.putText(frame, f"{label} {conf:.2f}",
                    (x1, y1 - 10),
                    cv2.FONT_HERSHEY_SIMPLEX, 0.5,
                    (0,255,0), 2)

        # Select best golf ball
        if label == "golf_ball" and conf > best_conf:
            best_conf = conf
            cx = int((x1 + x2) / 2)
            cy = int((y1 + y2) / 2)
            best_center = (cx, cy)

    # ----------------------------
    # Kalman: Predict
    # ----------------------------
    if initialized:
        prediction = kalman.predict()
        px = int(prediction[0].item())
        py = int(prediction[1].item())
    else:
        px, py = None, None

    # ----------------------------
    # Kalman: Update
    # ----------------------------
    if best_center is not None:
        measurement = np.array([
            [np.float32(best_center[0])],
            [np.float32(best_center[1])]
        ])

        if not initialized:
            kalman.statePre = np.array([
                [measurement[0][0]],
                [measurement[1][0]],
                [0],
                [0]
            ], np.float32)
            initialized = True

        kalman.correct(measurement)
        points.append(best_center)

    else:
        # No detection → use prediction
        if initialized and px is not None:
            points.append((px, py))

    # ----------------------------
    # Draw predicted point
    # ----------------------------
    if initialized and px is not None:
        cv2.circle(frame, (px, py), 6, (0,0,255), -1)

    # ----------------------------
    # Draw trajectory
    # ----------------------------
    for i in range(1, len(points)):
        if points[i-1] is None or points[i] is None:
            continue
        cv2.line(frame, points[i-1], points[i], (255,0,0), 2)

    # ----------------------------
    # Save + display
    # ----------------------------
    out.write(frame)
    cv2.imshow("Golf Ball Tracking", frame)

    if cv2.waitKey(1) & 0xFF == 27:
        break

# ----------------------------
# Cleanup
# ----------------------------
cap.release()
out.release()
cv2.destroyAllWindows()

In [ ]:
# Historical standalone prototype. The maintained app uses pipeline/.
from pathlib import Path
import sys
project_root = Path.cwd() if (Path.cwd() / "config.py").is_file() else Path.cwd().parent
sys.path.insert(0, str(project_root))
import config

import cv2
import numpy as np
from ultralytics import YOLO
from collections import deque
import os

# ----------------------------
# Load YOLOv8 model
# ----------------------------
model = YOLO(str(config.MODEL_PATH))

# ----------------------------
# Output folder (optional)
# ----------------------------
output_dir = str(config.OUTPUT_FOLDER / "prototypes")
os.makedirs(output_dir, exist_ok=True)

# ----------------------------
# Webcam input
# ----------------------------
cap = cv2.VideoCapture(0)

width  = int(cap.get(cv2.CAP_PROP_FRAME_WIDTH))
height = int(cap.get(cv2.CAP_PROP_FRAME_HEIGHT))
fps    = 30

# ----------------------------
# Save video (optional)
# ----------------------------
save_video = False  # set True to save

if save_video:
    output_path = os.path.join(output_dir, "live_output.avi")
    fourcc = cv2.VideoWriter_fourcc(*'XVID')
    out = cv2.VideoWriter(output_path, fourcc, fps, (width, height))
    print("Saving to:", output_path)

# ----------------------------
# Kalman Filter
# ----------------------------
kalman = cv2.KalmanFilter(4, 2)

kalman.measurementMatrix = np.array([
    [1, 0, 0, 0],
    [0, 1, 0, 0]
], np.float32)

kalman.transitionMatrix = np.array([
    [1, 0, 1, 0],
    [0, 1, 0, 1],
    [0, 0, 1, 0],
    [0, 0, 0, 1]
], np.float32)

kalman.processNoiseCov = np.eye(4, dtype=np.float32) * 0.03
kalman.measurementNoiseCov = np.eye(2, dtype=np.float32) * 0.5

# ----------------------------
# Tracking variables
# ----------------------------
points = deque(maxlen=30)
initialized = False
prediction = np.zeros((4,1), dtype=np.float32)

# ----------------------------
# Main loop
# ----------------------------
while True:
    ret, frame = cap.read()
    if not ret:
        break

    # ----------------------------
    # YOLO detection
    # ----------------------------
    results = model(frame, classes=[k for k, v in model.names.items() if v == "golf_ball"])[0]

    # ----------------------------
    # Keep only best box per class
    # ----------------------------
    best_per_class = {}

    for box in results.boxes:
        cls = int(box.cls[0])
        conf = float(box.conf[0])

        if conf < 0.2:
            continue

        if cls not in best_per_class or conf > best_per_class[cls][1]:
            best_per_class[cls] = (box, conf)

    # ----------------------------
    # Draw + select golf ball
    # ----------------------------
    best_center = None
    best_conf = 0

    for cls, (box, conf) in best_per_class.items():

        x1, y1, x2, y2 = map(int, box.xyxy[0])
        label = model.names[cls]

        # Draw box
        cv2.rectangle(frame, (x1, y1), (x2, y2), (0,255,0), 2)
        cv2.putText(frame, f"{label} {conf:.2f}",
                    (x1, y1 - 10),
                    cv2.FONT_HERSHEY_SIMPLEX, 0.5,
                    (0,255,0), 2)

        # Track only golf ball
        if label == "golf_ball" and conf > best_conf:
            best_conf = conf
            cx = int((x1 + x2) / 2)
            cy = int((y1 + y2) / 2)
            best_center = (cx, cy)

    # ----------------------------
    # Kalman Predict
    # ----------------------------
    if initialized:
        prediction = kalman.predict()
        px = int(prediction[0].item())
        py = int(prediction[1].item())
    else:
        px, py = None, None

    # ----------------------------
    # Kalman Update
    # ----------------------------
    if best_center is not None:
        measurement = np.array([
            [np.float32(best_center[0])],
            [np.float32(best_center[1])]
        ])

        if not initialized:
            kalman.statePre = np.array([
                [measurement[0][0]],
                [measurement[1][0]],
                [0],
                [0]
            ], np.float32)
            initialized = True

        kalman.correct(measurement)
        points.append(best_center)

    else:
        if initialized and px is not None:
            points.append((px, py))

    # ----------------------------
    # Draw prediction
    # ----------------------------
    if initialized and px is not None:
        cv2.circle(frame, (px, py), 6, (0,0,255), -1)

    # ----------------------------
    # Draw trajectory
    # ----------------------------
    for i in range(1, len(points)):
        if points[i-1] is None or points[i] is None:
            continue
        cv2.line(frame, points[i-1], points[i], (255,0,0), 2)

    # ----------------------------
    # Save + display
    # ----------------------------
    if save_video:
        out.write(frame)

    cv2.imshow("Live Golf Ball Tracking", frame)

    if cv2.waitKey(1) & 0xFF == 27:
        break

# ----------------------------
# Cleanup
# ----------------------------
cap.release()
if save_video:
    out.release()
cv2.destroyAllWindows()